# LIBRARY

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score 
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
#from lightgbm import LGBMClassifier
#from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from scipy.stats import randint, uniform
from collections import Counter
from collections import Counter
from imblearn.over_sampling import SMOTE
### Hyperparameter Tuning - XGBoost
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.metrics import roc_auc_score
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import randint, uniform
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint, uniform
import numpy as np



In [10]:
file_path = "../1.DATASET/CMI_FINAL_OD.csv"
df=pd.read_csv(file_path)

In [ ]:
column2drop = ['id']
#column2drop = ['id', 'sii', 'Basic_Demos-Sex', 'Physical-MAP_CAL', 'FGC-FGC_CORE_CAL', 'FGC-FGC_SR_CAL', 'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_Frame_num']

df.drop(column2drop, axis=1, inplace=True)
attributes = [col for col in df.columns if col != 'BIA-BIA_Fat']
X = df[attributes].values
y = np.array(df['BIA-BIA_Fat'])

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Classes : {np.unique(y)}")
print(f"Class counts : {dict(zip(*np.unique(y, return_counts=True)))}")

KeyError: 'sii'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# XGBoosting

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    'n_estimators': [100, 500, 1000],
    'learning_rate': [0.1, 0.01, 0.001, 0.0001],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'reg_alpha': [0.1, 0.01, 0.001],
    'reg_lambda': [0.1, 0.01, 0.001],
    'max_depth': [3, 5, 7],
    'max_leaves': [0, 32, 64],
    'n_jobs': [-1],
    'min_child_weight':  randint(1, 10),   # ← importante per classi rare
    'subsample':         uniform(0.7, 0.4),
    'colsample_bytree':  uniform(0.6, 0.4),
}

base_reg = XGBRegressor(
    objective='reg:squarederror',               
    tree_method='hist',             
    eval_metric='rmse',             
    random_state=42,
    n_jobs=-1,
)

rand_search = RandomizedSearchCV(
    estimator=base_reg,
    param_distributions=param_dist,
    cv=cv,
    scoring='r2', 
    n_iter=80,
    n_jobs=-1,
    refit=True,
    verbose=1,
    random_state=42,
)

In [ ]:
rand_search.fit(X_train, y_train) 

Fitting 5 folds for each of 80 candidates, totalling 400 fits


/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (

RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric='rmse',
                                          feature_types=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=N...
                                        'max_leaves': [0, 32, 64],
                                        'min_child_weight': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7f985d03ea90>,
                                        'n_estimators': [100, 500, 1000],
                                        'n_jobs': [-1],
                                        'reg_alpha': [0.1, 0.01, 0.001],
                                        'reg_lambda': [0.1, 0.01, 0.001],
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7f98597304c0>},
                   random_state=42, scoring='r2', verbose=1)

In [ ]:
print("Best params :", rand_search.best_params_)
print(f"Best CV RMSE: {rand_search.best_score_:.4f}")

best_model = rand_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"\nRMSE (test) : {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE  (test) : {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R²   (test) : {r2_score(y_test, y_pred):.4f}")

Best params : {'colsample_bytree': 0.9089273566942557, 'gamma': 0.2, 'learning_rate': 0.01, 'max_depth': 3, 'max_leaves': 32, 'min_child_weight': 5, 'n_estimators': 500, 'n_jobs': -1, 'reg_alpha': 0.01, 'reg_lambda': 0.001, 'subsample': 0.8613931464849588}
Best CV RMSE: 0.0815

RMSE (test) : 0.7194
MAE  (test) : 0.5619
R²   (test) : 0.0624
